# Get to Know a Dataset: elDORS v1 Database

This notebook serves as a guided tour of the [elDORS v1 Database](https://registry.opendata.aws/eldors-v1) dataset[cite: 3]. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/)[cite: 3].

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.[cite: 3]

At the top level of our S3 bucket, the 3.2 TB dataset is divided into four primary prefixes (distributions) to suit different computational environments:

 1. `elDORS_v1_raw/`: Baseline unclustered RNA sequence data in FASTA format.
 2. `GZIPPED_elDORS_v1/`: 80% sequence-identity clustered version provided in ~9GB sequence-aware chunks.
 3. `rMSA_optimized_elDORS/`: Optimized database build for the rMSA pipeline.
 4. `RNAcmap3_optimized_elDORS/`: Optimized split-strategy database (BLAST and Infernal formats) for the RNAcmap3 pipeline.
 
 Full documentation for this dataset can be found at: [Insert GitHub README Link]

In [ ]:
# This notebook requires the following additional libraries[cite: 3]
# (please install using the preferred method for your environment, e.g. pip, conda)[cite: 3]:
#
# boto3 >= 1.38.23[cite: 3]
# biopython >= 1.81
# matplotlib >= 3.10.3[cite: 3]

# Import the libraries required for this notebook[cite: 3]
import os
import boto3[cite: 3]
import matplotlib.pyplot as plt[cite: 3]
from botocore import UNSIGNED[cite: 3]
from botocore.config import Config[cite: 3]
from Bio import SeqIO

First, we will define the location of our dataset, create our boto3 S3 client, and list the top-level prefixes in our S3 bucket to verify the four distributions.

In [ ]:
# Location of the S3 bucket for this dataset[cite: 3]
bucket = "<PENDING-AWS-BUCKET-NAME>"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests[cite: 3].
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))[cite: 3]

# Print the items in the top-level prefixes[cite: 3]
print("Top-level distributions in elDORS:")
for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes'][cite: 3]:
    print(f"- {item['Prefix']}")[cite: 3]

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?[cite: 3]

Our dataset primarily consists of **FASTA** format files, along with specialized index formats like **BLAST** and **Infernal** databases. 

Generically, FASTA is a text-based format for representing either nucleotide sequences or amino acid sequences, in which base pairs or amino acids are represented using single-letter codes. It begins with a single-line description (starting with a `>`), followed by lines of sequence data.

Our dataset uses this format because:
 - It is the universal standard for bioinformatics and sequence alignment.
 - It easily handles massive biological sequences in a human-readable and machine-parsable format.
 
FASTA files are natively supported by almost all bioinformatics tools (like rMSA and RNAcmap3) and can be easily processed in Python using the `Biopython` library.

### Q: Can you show us an example of downloading and loading data from your dataset?[cite: 3]

As an example, let us download a single chunk from the clustered distribution to explore its contents locally. We will use the AWS CLI via a bash command for efficient downloading.

In [ ]:
%%bash
# Download a sample FASTA chunk (Note: replace 'chunk_001.fasta' with an actual filename from your bucket)
aws s3 cp s3://<PENDING-AWS-BUCKET-NAME>/GZIPPED_elDORS_v1/elDORS_v1_chunks/chunk_001.fasta.gz ./sample_data/ --no-sign-request

# Unzip for local analysis
gunzip ./sample_data/chunk_001.fasta.gz
ls -lh ./sample_data/

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

#### 1. Database Curation & Redundancy Removal Pipeline
The elDORS database is built by integrating vast genomic, non-coding RNA (ncRNA), and metagenomic resources. To make this resource computationally efficient, we applied a strict hierarchical redundancy removal and an 80% sequence identity clustering workflow:

![elDORS Curation Pipeline](../images/elDORS_creation_workflow.png)

#### 2. Downstream Optimization for Structure Prediction Pipelines
To enable immediate drop-in compatibility, the dataset is provided in specialized, pre-indexed builds optimized for cutting-edge RNA multiple sequence alignment (MSA) and contact prediction tools like `rMSA` and `RNAcmap3`:

![elDORS Downstream Pipelines](../images/elDORS_downstream_applications.png)

#### 3. Dynamic Inspection of a Downloaded Data Volume
Beyond the high-level architecture, users can dynamically parse individual volumes using Python. Below, we plot the sequence length distribution of the sample FASTA chunk we downloaded in the previous step.

In [ ]:
# Parse the FASTA file and extract sequence lengths
fasta_file = "./sample_data/chunk_001.fasta"
sequence_lengths = [len(record.seq) for record in SeqIO.parse(fasta_file, "fasta")]

# Plot using matplotlib
plt.figure(figsize=(10, 5), dpi=100, facecolor='white')

plt.hist(sequence_lengths, 
         bins=50,
         color='#2ecc71',
         edgecolor='white',
         linewidth=1.2,
         alpha=0.8)

plt.title('Distribution of RNA Sequence Lengths in Sample Chunk', 
         fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Sequence Length (Nucleotides)', fontsize=11, labelpad=10)
plt.ylabel('Count', fontsize=11, labelpad=10)

plt.grid(True, linestyle='--', alpha=0.3, color='gray')
ax = plt.gca()
ax.set_facecolor('#f8f9fa')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Total sequences analyzed in this sample volume: {len(sequence_lengths)}")

### Q: How is this dataset applied to downstream bioinformatics pipelines?

**Answer:** The elDORS database is explicitly engineered to fuel deep RNA Multiple Sequence Alignment (MSA). Our optimized builds act as drop-in replacements for legacy databases in `rMSA` and `RNAcmap3` pipelines. These deep MSAs are then utilized for Direct Coupling Analysis (DCA), 2D/3D structure prediction, and RNA language model (LM) training.

![Downstream Application](../images/elDORS_downstream_application.jpg)

Because institutional servers hosting legacy RNA databases are sometimes subject to regional network restrictions or severe latency timeouts, the AWS-hosted elDORS builds ensure researchers globally can execute these pipelines locally without interruption.

### Prerequisites: Tool Installation & Environment Setup

The elDORS dataset provides the massive, optimized sequence infrastructure required for deep RNA MSA generation. However, users are responsible for independently installing, configuring, and activating the downstream pipelines in their local environments (e.g., via Conda). 

Please refer to the official repositories for installation instructions, and ensure you cite the original authors when utilizing their tools:

*   **RNAcmap3-elDORS Pipeline:** The RNAcmap3 pipeline relies on the [RNAcmap2 search architecture](https://github.com/jaswindersingh2/RNAcmap2) but traditionally utilizes the MARS database. By replacing MARS with our optimized builds, you are executing the **RNAcmap3-elDORS** pipeline benchmarked in our manuscript.
*   **rMSA:** [pylelab/rMSA] (https://github.com/pylelab/rMSA)

### Integrating elDORS into Standard Pipelines

To use the optimized elDORS builds with your existing pipelines, you do not need to reinstall the underlying tools. You simply need to point the pipeline's database variable to your downloaded AWS S3 directory.

#### For rMSA Users: Modifying the Wrapper Script
If you are using the standard `rMSA` wrapper scripts, open the main execution script (e.g., `rMSA.pl` or the equivalent configuration file) in a text editor. 

Locate the database directory variables at the top of the script. You will keep your standard Rfam and RNAcentral paths, but you will update the main sequence database variable (often `$db2` or `$dbnewdir`) to point to your downloaded `rMSA_optimized_elDORS` path.

**Updated Config for elDORS (`rMSA.pl`):**
```perl
#!/usr/bin/perl
use strict;
use File::Basename;
use Cwd 'abs_path';

my $rootdir=dirname(abs_path(__FILE__));
my $bindir ="$rootdir/bin";
my $dbdir  ="$rootdir/database";

# Point to your downloaded AWS elDORS directory
my $dbnewdir ="/absolute/path/to/rMSA_optimized_elDORS"; 

my $db0    ="$dbdir/Rfam.cm";
my $db1    ="$dbdir/rnacentral.fasta";

# Target the elDORS v1 database prefix
my $db2    ="$dbnewdir/elDORS_v1_db"; 

# Note: If your specific rMSA pipeline flavor requires the raw, uncompressed 
# FASTA file instead of the BLAST database prefix, use:
# my $db2  ="$dbnewdir/elDORS_v1";

#### For RNAcmap3 Users: Creating a Run Script
RNAcmap3 utilizes a split-strategy that requires both a BLAST-formatted database and an Infernal-compatible sequence database. The `RNAcmap3_optimized_elDORS` distribution is pre-packaged with both. 

You can use the following generalized bash template to execute RNAcmap3 by pointing the `-b` (BLAST) and `-c` (Infernal) flags to your downloaded directories.



**Generalized RNAcmap3-elDORS Execution Template:**
```bash
#!/bin/bash
# Generalized template for running RNAcmap3 with elDORS v1

# 0. Activate the rna_cmap2 environment
# conda activate rna_cmap2

# 1. Set your input sequence and computational resources
INPUT_FASTA="your_target_sequence.fasta"
THREADS=8 #change the number of threads as per requirement

# 2. Point to your downloaded AWS elDORS directories
# NOTE: Update "/absolute/path/to/" to match your actual download location
ELDORS_DIR="/absolute/path/to/RNAcmap3_optimized_elDORS"

# The BLAST database prefix inside the blast/ directory
BLAST_DB="${ELDORS_DIR}/blast/elDORS_v1_db"

# The Infernal directory containing the uncompressed FASTA volumes
INFERNAL_DB="${ELDORS_DIR}/infernal"

# 3. Execute the pipeline
# -i: Input FASTA
# -n: Number of threads
# -b: BLAST database path
# -c: Infernal database path
# Add additional flags (e.g., -d gremlin) as required by your specific workflow
bash /path/to/your/RNAcamp3/run_rnacmap.sh \
    -i "$INPUT_FASTA" \
    -n "$THREADS" \
    -b "$BLAST_DB" \
    -c "$INFERNAL_DB"